In [ ]:
import json

import pandas as pd
import great_expectations as gx
import synapseclient

from agoradatatools.gx import GreatExpectationsRunner

context = gx.get_context(project_root_dir='../src/agoradatatools/great_expectations')

from expectations.expect_column_values_to_have_list_members_of_type import ExpectColumnValuesToHaveListMembersOfType
from expectations.expect_column_values_to_have_list_length_in_range import ExpectColumnValuesToHaveListLengthInRange
from expectations.expect_column_values_to_have_list_members import ExpectColumnValuesToHaveListMembers

# Create Expectation Suite for Nominated Targets Data

## Get Example Data File

In [ ]:
syn = synapseclient.Synapse()
syn.login()

In [ ]:
# Synapse ID of the processed nominated_targets output file.
# Alternatively, use the local staging file: nominated_targets_file = "../staging/nominated_targets.json"
nominated_targets_file = syn.get("syn73695285").path

## Create Validator Object on Data File

In [ ]:
# nominated_targets has no dict/link columns, so no columns are passed to
# convert_nested_columns_to_json. The list columns (nominating_teams,
# cohort_studies, input_data, programs) stay as Python lists so the custom list
# expectations can validate them directly.
df = pd.read_json(nominated_targets_file)
validator = context.sources.pandas_default.read_dataframe(df)
validator.expectation_suite_name = "nominated_targets"

## Add Expectations to Validator Object For Each Column

In [ ]:
# ensembl_gene_id
validator.expect_column_values_to_be_of_type("ensembl_gene_id", "str")
validator.expect_column_values_to_not_be_null("ensembl_gene_id")
validator.expect_column_values_to_be_unique("ensembl_gene_id")
validator.expect_column_values_to_match_regex("ensembl_gene_id", regex=r"^ENSG\d+$")

In [ ]:
# hgnc_symbol (renamed from symbol via agora_rename)
validator.expect_column_values_to_be_of_type("hgnc_symbol", "str")
validator.expect_column_values_to_not_be_null("hgnc_symbol")

In [ ]:
# total_nominations
validator.expect_column_values_to_be_of_type("total_nominations", "int")
validator.expect_column_values_to_not_be_null("total_nominations")
validator.expect_column_values_to_be_between("total_nominations", min_value=1)

In [ ]:
# initial_nomination (nullable year; be_between skips nulls)
validator.expect_column_values_to_be_between("initial_nomination", min_value=2000, max_value=2100)

In [ ]:
# nominating_teams
validator.expect_column_values_to_be_of_type("nominating_teams", "list")
validator.expect_column_values_to_not_be_null("nominating_teams")
validator.expect_column_values_to_have_list_length_in_range(column="nominating_teams", list_length_range=[1, 100])
validator.expect_column_values_to_have_list_members_of_type(column="nominating_teams", member_type="str")
validator.expect_column_values_to_have_list_members(column="nominating_teams", list_members=["ASU", "Chang Lab", "Columbia", "Columbia-Rush", "Duke", "Duke BARU", "Emory", "Emory-Sage-SGC", "Harvard-MIT", "IUSM-Purdue", "JAX-VUMC-UW Resilience", "Longo Lab", "Mayo", "Mayo-UFL-ISB", "MSSM - Roussos Lab", "MSSM - Zhang Lab"])

In [ ]:
# cohort_studies
# Min length is 0: some nominated genes legitimately have no recorded cohort study,
# which the transform represents as an empty list.
validator.expect_column_values_to_be_of_type("cohort_studies", "list")
validator.expect_column_values_to_not_be_null("cohort_studies")
validator.expect_column_values_to_have_list_length_in_range(column="cohort_studies", list_length_range=[0, 100])
validator.expect_column_values_to_have_list_members_of_type(column="cohort_studies", member_type="str")

In [ ]:
# input_data
validator.expect_column_values_to_be_of_type("input_data", "list")
validator.expect_column_values_to_not_be_null("input_data")
validator.expect_column_values_to_have_list_length_in_range(column="input_data", list_length_range=[1, 100])
validator.expect_column_values_to_have_list_members_of_type(column="input_data", member_type="str")
validator.expect_column_values_to_have_list_members(column="input_data", list_members=["Behavior", "Clinical", "Genetics", "Metabolomics", "Pathway Modeling", "Phenomics", "Protein", "Risk Scores", "RNA", "Unknown"])

In [ ]:
# programs
validator.expect_column_values_to_be_of_type("programs", "list")
validator.expect_column_values_to_not_be_null("programs")
validator.expect_column_values_to_have_list_length_in_range(column="programs", list_length_range=[1, 100])
validator.expect_column_values_to_have_list_members_of_type(column="programs", member_type="str")
validator.expect_column_values_to_have_list_members(column="programs", list_members=["AMP-AD", "Community", "FunGen-AD", "Resilience-AD", "TREAT-AD"])

In [ ]:
# pharos_class (nullable; be_in_set skips nulls)
validator.expect_column_values_to_be_in_set("pharos_class", {"Tclin", "Tchem", "Tbio", "Tdark"})

## Save Expectation Suite

In [ ]:
validator.save_expectation_suite(discard_failed_expectations=False)

## Create Checkpoint and View Results

In [ ]:
checkpoint = context.add_or_update_checkpoint(
    name="agora-test-checkpoint",
    validator=validator,
)
checkpoint_result = checkpoint.run()
context.view_validation_result(checkpoint_result)

## Build Data Docs - Click on Expectation Suite to View All Expectations

In [ ]:
context.build_data_docs()
context.open_data_docs()